In [1]:
import sys
import os
sys.path.append(os.pardir)

In [2]:
import pandas as pd

In [3]:
import numpy as np

In [4]:
#df = pd.read_csv("../data/raw/tempload.csv", index_col = "date", parse_dates = True)
#need to drop duplicates
#idk why the extract script is buggy
#chain it all togther
# df isn't defined yet in the df["date"] call so we need to use pipe
df = (pd.read_csv("../data/raw/tempload.csv")
        .drop_duplicates()
        .pipe(lambda df: df.set_index(pd.to_datetime(df["date"])))
        .drop(columns = ["date"])
     )
#this will throw an error if there are duplicates
df.index.freq = 'h'

In [10]:
# check for missing values
start_date = df.index.min()
end_date = df.index.max()
complete_date_range = pd.date_range(start=start_date, end=end_date, freq=df.index.freq)
is_index_complete = (df.index == complete_date_range).all()
print(f"Index complete: {is_index_complete}")
print(f"Number of rows with missing values: {df.isnull().any(axis=1).mean()}")

Index complete: True
Number of rows with missing values: 0.0009504257907542579


In [18]:
df[df.isnull().any(axis = 1)]

,value,temperature,is_day
date,,,
2024-02-20 07:00:00+00:00,NaN,3.061500,0.0
2024-02-20 08:00:00+00:00,NaN,2.061500,0.0
2024-02-20 09:00:00+00:00,NaN,1.811500,0.0
2024-02-20 10:00:00+00:00,NaN,1.161500,0.0
2024-02-20 11:00:00+00:00,NaN,1.061500,0.0
2024-02-20 12:00:00+00:00,NaN,0.611500,0.0
2024-02-20 13:00:00+00:00,NaN,-0.288500,0.0
2024-02-20 14:00:00+00:00,NaN,1.011500,1.0
2024-02-20 15:00:00+00:00,NaN,5.011500,1.0


In [9]:
df.index

DatetimeIndex(['2023-01-01 00:00:00+00:00', '2023-01-01 01:00:00+00:00',
               '2023-01-01 02:00:00+00:00', '2023-01-01 03:00:00+00:00',
               '2023-01-01 04:00:00+00:00', '2023-01-01 05:00:00+00:00',
               '2023-01-01 06:00:00+00:00', '2023-01-01 07:00:00+00:00',
               '2023-01-01 08:00:00+00:00', '2023-01-01 09:00:00+00:00',
               ...
               '2025-12-31 14:00:00+00:00', '2025-12-31 15:00:00+00:00',
               '2025-12-31 16:00:00+00:00', '2025-12-31 17:00:00+00:00',
               '2025-12-31 18:00:00+00:00', '2025-12-31 19:00:00+00:00',
               '2025-12-31 20:00:00+00:00', '2025-12-31 21:00:00+00:00',
               '2025-12-31 22:00:00+00:00', '2025-12-31 23:00:00+00:00'],
              dtype='datetime64[us, UTC]', name='date', length=26304, freq='h')

## US Holidays

In [15]:
from pandas.tseries.holiday import USFederalHolidayCalendar as calendar

In [17]:
cal = calendar()
holidays = cal.holidays(start = df.index.min(), end = df.index.max())

In [21]:
holidays

DatetimeIndex(['2023-01-02 00:00:00+00:00', '2023-01-16 00:00:00+00:00',
               '2023-02-20 00:00:00+00:00', '2023-05-29 00:00:00+00:00',
               '2023-06-19 00:00:00+00:00', '2023-07-04 00:00:00+00:00',
               '2023-09-04 00:00:00+00:00', '2023-10-09 00:00:00+00:00',
               '2023-11-10 00:00:00+00:00', '2023-11-23 00:00:00+00:00',
               '2023-12-25 00:00:00+00:00', '2024-01-01 00:00:00+00:00',
               '2024-01-15 00:00:00+00:00', '2024-02-19 00:00:00+00:00',
               '2024-05-27 00:00:00+00:00', '2024-06-19 00:00:00+00:00',
               '2024-07-04 00:00:00+00:00', '2024-09-02 00:00:00+00:00',
               '2024-10-14 00:00:00+00:00', '2024-11-11 00:00:00+00:00',
               '2024-11-28 00:00:00+00:00', '2024-12-25 00:00:00+00:00',
               '2025-01-01 00:00:00+00:00', '2025-01-20 00:00:00+00:00',
               '2025-02-17 00:00:00+00:00', '2025-05-26 00:00:00+00:00',
               '2025-06-19 00:00:00+00:00', '2025-0

In [19]:
df["Holiday"]= df.index.isin(holidays)

In [ ]:
#check it
df.loc[df.index.isin(holidays)]

In [35]:
df.drop(columns = ["Holiday"])

,value,temperature
date,,
2023-01-01 00:00:00+00:00,28291.0,11.361500
2023-01-01 01:00:00+00:00,28727.0,9.561500
2023-01-01 02:00:00+00:00,28370.0,7.961500
2023-01-01 03:00:00+00:00,27955.0,7.311500
2023-01-01 04:00:00+00:00,27668.0,6.261500
...,...,...
2025-12-31 19:00:00+00:00,32891.0,8.911500
2025-12-31 20:00:00+00:00,32281.0,10.361500
2025-12-31 21:00:00+00:00,31630.0,10.761499


In [33]:
#chain it in a pipe
def set_holidays(df):
    cal = calendar()
    holidays = cal.holidays(start = df.index.min(), end = df.index.max())
    df["Holiday"]= df.index.isin(holidays)
    return df

In [36]:
df = df.pipe(set_holidays)

In [37]:
df.head()

,value,temperature,Holiday
date,,,
2023-01-01 00:00:00+00:00,28291.0,11.3615,False
2023-01-01 01:00:00+00:00,28727.0,9.5615,False
2023-01-01 02:00:00+00:00,28370.0,7.9615,False
2023-01-01 03:00:00+00:00,27955.0,7.3115,False
2023-01-01 04:00:00+00:00,27668.0,6.2615,False


In [ ]:
"""
dr = pd.date_range(start='2015-07-01', end='2015-07-31')
df = pd.DataFrame()
df['Date'] = dr

cal = calendar()
holidays = cal.holidays(start=dr.min(), end=dr.max())

df['Holiday'] = df['Date'].isin(holidays)
print df
"""

### Calendar Features

In [23]:
df["month"] = df.index.month
df["week"] = df.index.isocalendar().week
df["day_of_week"] = df.index.day_of_week
df["hour"]=df.index.hour

In [24]:
df.head()

,value,temperature,year,day_of_week,hour,month,week
date,,,,,,,
2023-01-01 00:00:00+00:00,28291.0,11.3615,2023,6,0,1,52
2023-01-01 01:00:00+00:00,28727.0,9.5615,2023,6,1,1,52
2023-01-01 02:00:00+00:00,28370.0,7.9615,2023,6,2,1,52
2023-01-01 03:00:00+00:00,27955.0,7.3115,2023,6,3,1,52
2023-01-01 04:00:00+00:00,27668.0,6.2615,2023,6,4,1,52


In [34]:
#wrap it up to pipe it
def cal_features(df):
    df = df.assign(month = lambda df: df.index.month,
              week = lambda df: df.index.isocalendar().week, 
              day_of_week = lambda df: df.index.day_of_week,
              hour = lambda df: df.index.hour
             )
    return df

In [45]:
df = df.pipe(cal_features)

In [46]:
df.head()

,value,temperature,month,week,day_of_week,hour
date,,,,,,
2023-01-01 00:00:00+00:00,28291.0,11.3615,1,52,6,0
2023-01-01 01:00:00+00:00,28727.0,9.5615,1,52,6,1
2023-01-01 02:00:00+00:00,28370.0,7.9615,1,52,6,2
2023-01-01 03:00:00+00:00,27955.0,7.3115,1,52,6,3
2023-01-01 04:00:00+00:00,27668.0,6.2615,1,52,6,4


## Cyclic Features

In [41]:
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["week_sin"] = np.sin(2 * np.pi * df["week"] / 52)
df["week_cos"] = np.cos(2 * np.pi * df["week"] / 52)

df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

In [42]:
df.head()

,value,temperature,month,week,day_of_week,hour,month_sin,month_cos,week_sin,week_cos,day_sin,day_cos,hour_sin,hour_cos
date,,,,,,,,,,,,,,
2023-01-01 00:00:00+00:00,28291.0,11.3615,1,52,6,0,0.5,0.866025,0.0,1.0,-0.781831,0.62349,0.000000,1.000000
2023-01-01 01:00:00+00:00,28727.0,9.5615,1,52,6,1,0.5,0.866025,0.0,1.0,-0.781831,0.62349,0.258819,0.965926
2023-01-01 02:00:00+00:00,28370.0,7.9615,1,52,6,2,0.5,0.866025,0.0,1.0,-0.781831,0.62349,0.500000,0.866025
2023-01-01 03:00:00+00:00,27955.0,7.3115,1,52,6,3,0.5,0.866025,0.0,1.0,-0.781831,0.62349,0.707107,0.707107
2023-01-01 04:00:00+00:00,27668.0,6.2615,1,52,6,4,0.5,0.866025,0.0,1.0,-0.781831,0.62349,0.866025,0.500000


In [43]:
def cyclic_features(df):
    df = df.assign(
        month_sin = lambda df: np.sin(2 * np.pi * df["month"] / 12),
        month_cos = lambda df: np.cos(2 * np.pi * df["month"] / 12),
        week_sin = lambda df: np.sin(2 * np.pi * df["week"] / 52),
        week_cos = lambda df: np.cos(2 * np.pi * df["week"] / 52),
        day_sin = lambda df: np.sin(2 * np.pi * df["day_of_week"] / 7),
        day_cos = lambda df: np.cos(2 * np.pi * df["day_of_week"] / 7),
        hour_sin = lambda df: np.sin(2 * np.pi * df["hour"] / 24),
        hour_cos = lambda df: np.cos(2 * np.pi * df["hour"] / 24)
    )
    return df

In [47]:
df = df.pipe(cyclic_features)

In [48]:
df.head()

,value,temperature,month,week,day_of_week,hour,month_sin,month_cos,week_sin,week_cos,day_sin,day_cos,hour_sin,hour_cos
date,,,,,,,,,,,,,,
2023-01-01 00:00:00+00:00,28291.0,11.3615,1,52,6,0,0.5,0.866025,0.0,1.0,-0.781831,0.62349,0.000000,1.000000
2023-01-01 01:00:00+00:00,28727.0,9.5615,1,52,6,1,0.5,0.866025,0.0,1.0,-0.781831,0.62349,0.258819,0.965926
2023-01-01 02:00:00+00:00,28370.0,7.9615,1,52,6,2,0.5,0.866025,0.0,1.0,-0.781831,0.62349,0.500000,0.866025
2023-01-01 03:00:00+00:00,27955.0,7.3115,1,52,6,3,0.5,0.866025,0.0,1.0,-0.781831,0.62349,0.707107,0.707107
2023-01-01 04:00:00+00:00,27668.0,6.2615,1,52,6,4,0.5,0.866025,0.0,1.0,-0.781831,0.62349,0.866025,0.500000


## Sunlight features

In [37]:
# TODO